# Project Name - Online Retail Customer Behavior  Analytics: A Power BI Dashboard Project

## Group Members : Isha Patel, Yashvi Soni, Sakshi Sharma, Rishi Kanadia 

## Project brief

### Project Goal
This project turns online retail transaction data into a simple story about customers. It helps businesses understand who their best customers are, who is about to leave, and how to keep them coming back.

### Objective
Build an easy-to-use dashboard that shows customer behavior, finds patterns, and helps the business make smarter decisions to increase sales and loyalty.

### Key Project Details:
--> Customers are grouped into types: Champions (best), Loyal, At Risk (might leave), and Hibernating (already left).

--> Track how many customers return month after month using cohort analysis.

--> Find out which customers spend the most and which products sell best together.

--> See which countries bring the most revenue.

--> Get clear recommendations: who to reward, who to win back, and where to focus marketing.

### Dataset
We are going to use classic Online Retail dataset (UK-based, 2010-2011) containing invoices, products, customers, and countries. It's messy, real-world data – perfect for a detective story.

### Importing the libraries

In [2]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

# Set up visual style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

### Importing the dataset

In [3]:
df = pd.read_csv('Online Retail.csv')

In [4]:
df.head()

,index,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [5]:
df.tail()

,index,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
541904,541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,12/9/2011 12:50,0.85,12680.0,France
541905,541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,12/9/2011 12:50,2.10,12680.0,France
541906,541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,12/9/2011 12:50,4.15,12680.0,France
541907,541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,12/9/2011 12:50,4.15,12680.0,France
541908,541908,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,12/9/2011 12:50,4.95,12680.0,France


### Exploring the dataset using different function

In [6]:
df.shape

(541909, 9)

In [7]:
df.describe()

,index,Quantity,UnitPrice,CustomerID
count,541909.00000,541909.000000,541909.000000,406829.000000
mean,270954.00000,9.552250,4.611114,15287.690570
std,156435.79785,218.081158,96.759853,1713.600303
min,0.00000,-80995.000000,-11062.060000,12346.000000
25%,135477.00000,1.000000,1.250000,13953.000000
50%,270954.00000,3.000000,2.080000,15152.000000
75%,406431.00000,10.000000,4.130000,16791.000000
max,541908.00000,80995.000000,38970.000000,18287.000000


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   index        541909 non-null  int64  
 1   InvoiceNo    541909 non-null  object 
 2   StockCode    541909 non-null  object 
 3   Description  540455 non-null  object 
 4   Quantity     541909 non-null  int64  
 5   InvoiceDate  541909 non-null  object 
 6   UnitPrice    541909 non-null  float64
 7   CustomerID   406829 non-null  float64
 8   Country      541909 non-null  object 
dtypes: float64(2), int64(2), object(5)
memory usage: 37.2+ MB


In [9]:
missing = df.isnull().sum()
missing

index               0
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

### Cleaning the dataset 

In [10]:
df_clean = df.dropna(subset=['CustomerID']).copy()

In [11]:
df_clean = df_clean[df_clean['Quantity'] > 0]

In [12]:
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C', na=False)]

In [21]:
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int).astype(str)

In [24]:
if 'InvoiceDateTime' in df_clean.columns:
    # Already exists – ensure it's datetime
    df_clean['InvoiceDateTime'] = pd.to_datetime(df_clean['InvoiceDateTime'], errors='coerce')
    print("Using existing 'InvoiceDateTime' column.")

elif 'InvoiceDate' in df_clean.columns and 'InvoiceTime' in df_clean.columns:
    # Both date and time columns exist – combine them
    df_clean['InvoiceDateTime'] = pd.to_datetime(
        df_clean['InvoiceDate'].astype(str) + ' ' + df_clean['InvoiceTime'].astype(str),
        errors='coerce'
    )
    # Drop the original separate columns
    df_clean.drop(columns=['InvoiceDate', 'InvoiceTime'], inplace=True)
    print("Combined 'InvoiceDate' and 'InvoiceTime' into 'InvoiceDateTime'.")

elif 'InvoiceDate' in df_clean.columns:
    # Only InvoiceDate – assume it already includes time (or convert and keep)
    df_clean['InvoiceDateTime'] = pd.to_datetime(df_clean['InvoiceDate'], errors='coerce')
    # Optionally drop the original column if you want only the combined one
    df_clean.drop(columns=['InvoiceDate'], inplace=True)
    print("Converted 'InvoiceDate' to datetime and stored as 'InvoiceDateTime'.")

else:
    # No suitable date column found – warn user
    print("Warning: No date/time column found to create 'InvoiceDateTime'.")

Converted 'InvoiceDate' to datetime and stored as 'InvoiceDateTime'.


In [25]:
print(f"\nRemaining rows: {len(df_clean)}")
print(f"Unique customers: {df_clean['CustomerID'].nunique()}")
print("\nData types after cleaning:")
print(df_clean.dtypes)


Remaining rows: 397924
Unique customers: 4339

Data types after cleaning:
index                       int64
InvoiceNo                  object
StockCode                  object
Description                object
Quantity                    int64
UnitPrice                 float64
CustomerID                 object
Country                    object
InvoiceDateTime    datetime64[ns]
dtype: object


In [27]:
df_clean.isnull().sum()

index              0
InvoiceNo          0
StockCode          0
Description        0
Quantity           0
UnitPrice          0
CustomerID         0
Country            0
InvoiceDateTime    0
dtype: int64